In [2]:
%restart_python
!pip install tldextract


UsageError: Line magic function `%restart_python` not found.


In [3]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
import tldextract


def extract_domain_parts(url):
    """Extract domain parts using tldextract"""
    try:
        ext = tldextract.extract(url)
        registered_domain = '.'.join([part for part in [ext.domain, ext.suffix] if part])
        return registered_domain if registered_domain else None  
    except Exception as e:
        return None

# Register UDF
extract_domain_udf = udf(extract_domain_parts, StringType())

In [4]:
# Read Site Visit data
df_site_visit = spark \
    .read \
    .parquet(f"s3://mntn-data-archive-prod/signals/site_visit_signal/dt=2025-07-23/hh=00/") 

#Aggregating Site Visit data
site_visit_domains = df_site_visit \
    .withColumn("domain", extract_domain_udf("url")) \
    .filter(F.col("domain").isNotNull() & F.col("url").isNotNull()) \
    .groupBy("domain") \
    .agg(
        F.first("url").alias("url"),
        F.count("*").alias("total_count")) \
    .orderBy(F.desc("total_count"))

# Cache it
site_visit_domains.cache()
print(f"Site visit domains: {site_visit_domains.count():,}")

# Show sample to verify extraction worked
print("\nSample extracted domains:")
site_visit_domains.show(20, truncate=True)



NameError: name 'spark' is not defined